# 🛡️ LakeLogic: The 5-Minute Data Contract Quickstart

Welcome to LakeLogic! In this tutorial, you'll learn how to move from a **messy CSV export** to a **validated, production-ready Silver table** in minutes using declarative data contracts.

### 🎯 The Goal
We have a CSV export from a CRM that has duplicates, invalid emails, negative spending, and inconsistent types. We need to:
1. **Deduplicate** records.
2. **Enforce** a strict schema.
3. **Validate** business rules (no negative spending).
4. **Transform** the data (rename columns).
5. **Quarantine** anything that doesn't fit.

Let's get started!

## 1. Setup
We'll start by importing the `DataProcessor`. We'll also tell LakeLogic to use the **Polars** engine for local speed.

In [1]:
import os
import polars as pl
from IPython.display import display
from lakelogic import DataProcessor

# Force local processing with Polars
os.environ["LAKELOGIC_ENGINE"] = "duckdb"

def show_df(df, title=None, max_rows=100):
    if title:
        print(title)
    if os.environ.get("LAKELOGIC_ENGINE") == "spark":
        df.show(max_rows, False)
    else:
        display(df)

def count_rows(df):
    if os.environ.get("LAKELOGIC_ENGINE") == "spark":
        return df.count()
    return len(df)


## 2. Peek at the Messy Data
Let's look at the raw contents of `data/crm_export.csv`. Note the issues:
- Customer `104` appearing twice (duplicate).
- Customer `102` having an invalid email format.
- Customer `103` missing an email entirely.
- Customer `105` having negative spend.
- Customer `107` having non-numeric spend data ("high").

In [2]:
df = pl.read_csv("data/crm_export.csv")
show_df(df, "RAW DATA:")


RAW DATA:


customer_id,name,email,signup_date,plan_type,total_spend,is_active
i64,str,str,str,str,str,bool
101,"""Alice Smith""","""alice.smith@example.com""","""2024-01-15""","""premium""","""1250.50""",true
102,"""Bob Jones""","""bob.jones_invalid_email""","""2024-01-16""","""basic""","""45.00""",true
103,"""Charlie Brown""",null,"""2024-01-17""","""free""","""0""",true
104,"""Duplicate User""","""dup@example.com""","""2024-01-15""","""free""","""10.00""",true
104,"""Duplicate User""","""dup@example.com""","""2024-01-18""","""free""","""25.00""",true
105,"""Eve Adams""","""eve@example.com""","""2024-02-01""","""premium""","""-500""",false
106,"""Frank White""","""frank@example.com""","""not-a-date""","""basic""","""100""",true
107,"""Grace Lee""","""grace@example.com""","""2024-02-10""","""gold""","""high""",true


## 3. The Multi-Stage Contract
LakeLogic allows you to run the **same contract** in different "stages". 

Take a moment to open `contract.yaml` in your editor. You'll see the **Stages** defined at the top:
- **Bronze Stage**: Ingests raw data using a glob pattern (`data/crm_*.csv`), deduplicates, and renames columns.
- **Silver Stage (Default)**: Consumes the Bronze Parquet output, renames fields, deduplicates, and enforces strict business quality rules.

Run logs are written to logs/lakelogic_run_logs.duckdb (configured in metadata) and include max_source_mtime and source_files_json for incremental watermarking.

By putting stages at the top, the contract becomes a self-documenting pipeline blueprint.

## 4. Execute Step 1: Bronze Ingestion
Notice how we don't need to specify a path here! `DataProcessor` automatically pulls the correct path from the `bronze` stage definition.

In [3]:
bronze_proc = DataProcessor(contract="contract.yaml", stage="bronze")
raw_df_b, good_df_b, bad_df_b = bronze_proc.run_source()
bronze_proc.materialize(good_df_b, bad_df_b)

print(f"Bronze records processed: {count_rows(good_df_b)}")
show_df(good_df_b, "BRONZE OUTPUT:")


2026-02-08 10:21:06.313 | INFO     | lakelogic.core.processor:run_source:399 - Loading source: data\crm_*.csv via duckdb
2026-02-08 10:21:06.831 | INFO     | lakelogic.core.processor:run:278 - Starting LakeLogic run [Auto-Engine: duckdb, Contract: Validated CRM customers]
2026-02-08 10:21:06.883 | INFO     | lakelogic.core.processor:run:324 - Run complete. [domain=customer_analytics, system=crm_export] Source: 8, Total: 8, Pre-Transform Dropped: 0
2026-02-08 10:21:06.978 | INFO     | lakelogic.core.materialization:_write_run_log_table:1703 - Wrote run log to DuckDB table lakelogic_run_logs (logs\lakelogic_run_logs.duckdb)
2026-02-08 10:21:07.035 | INFO     | lakelogic.core.materialization:materialize_dataframe:997 - Materialized 32 rows to data\bronze\bronze_customers.parquet


Bronze records processed: 8
BRONZE OUTPUT:


,customer_id,name,email,signup_date,plan_type,total_spend,is_active,_lakelogic_source,_lakelogic_processed_at,_lakelogic_run_id,_lakelogic_domain,_lakelogic_system
0,101,Alice Smith,alice.smith@example.com,2024-01-15,premium,1250.50,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
1,102,Bob Jones,bob.jones_invalid_email,2024-01-16,basic,45.00,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
2,103,Charlie Brown,NaN,2024-01-17,free,0,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
3,104,Duplicate User,dup@example.com,2024-01-15,free,10.00,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
4,104,Duplicate User,dup@example.com,2024-01-18,free,25.00,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
5,105,Eve Adams,eve@example.com,2024-02-01,premium,-500,False,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
6,106,Frank White,frank@example.com,not-a-date,basic,100,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export
7,107,Grace Lee,grace@example.com,2024-02-10,gold,high,True,data\crm_*.csv,2026-02-08T10:21:06+00:00,ec03d9177409485381838c0ef6a90506,customer_analytics,crm_export


## 5. Execute Step 2: Silver Quality Gate
Now we run the default (Silver) stage. It uses the `source.path` defined for the Silver layer, which points to the files we just created in Bronze.

In [13]:
silver_proc = DataProcessor(contract="contract.yaml")
bronze_df, good_df, bad_df = silver_proc.run_source()

x= silver_proc.materialize(good_df, bad_df)

2026-02-08 12:18:47.926 | INFO     | lakelogic.core.processor:run_source:399 - Loading source: D:\Github\_SaaS\lakelogic\examples\02_tutorials\medallion_architecture\data\bronze\bronze_customers.parquet via duckdb
2026-02-08 12:18:47.931 | INFO     | lakelogic.core.processor:run:278 - Starting LakeLogic run [Auto-Engine: duckdb, Contract: Validated CRM customers]


2026-02-08 12:18:48.015 | INFO     | lakelogic.core.processor:run:319 - Run complete. [domain=customer_analytics, system=crm_export] Source: 32, Total (post-transform): 7, Good: 2, Quarantined: 5, Pre-Transform Dropped: 25, Ratio: 71.43%
2026-02-08 12:18:48.091 | INFO     | lakelogic.core.materialization:_write_run_log_table:1703 - Wrote run log to DuckDB table lakelogic_run_logs (logs\lakelogic_run_logs.duckdb)
2026-02-08 12:18:48.142 | INFO     | lakelogic.core.materialization:materialize_dataframe:997 - Materialized 2 rows to data\silver\silver_customers.parquet


In [14]:
x


{'target': 'data\\silver\\silver_customers.parquet',
 'rows_written': 2,
 'format': 'parquet'}

## 6. Inspect Results

### The 'Good' Data (Silver Table)
Only records that passed **every** rule made it here. Notice that types are now correct and business logic is satisfied.

In [6]:
show_df(good_df, "PRODUCTION-READY RECORDS:")

PRODUCTION-READY RECORDS:


┌─────────────┬────────────────┬─────────────────────────┬─────────────┬─────────┬─────────────┬───────────┐
│ customer_id │      name      │          email          │ signup_date │  tier   │ total_spend │ is_active │
│    int64    │    varchar     │         varchar         │    date     │ varchar │   double    │  boolean  │
├─────────────┼────────────────┼─────────────────────────┼─────────────┼─────────┼─────────────┼───────────┤
│         101 │ Alice Smith    │ alice.smith@example.com │ 2024-01-15  │ premium │      1250.5 │ true      │
│         104 │ Duplicate User │ dup@example.com         │ 2024-01-18  │ free    │        25.0 │ true      │
└─────────────┴────────────────┴─────────────────────────┴─────────────┴─────────┴─────────────┴───────────┘

### The 'Bad' Data (Quarantine)
Records that failed are isolated with detailed reasons in `_lakelogic_errors`.

In [7]:
show_df(bad_df, "QUARANTINED RECORDS:")

QUARANTINED RECORDS:


┌─────────────┬───────────────┬─────────────────────────┬─────────────┬─────────┬─────────────┬───────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────┬──────────────────┬────────────────────────┐
│ customer_id │     name      │          email          │ signup_date │  tier   │ total_spend │ is_active │                                                                            _lakelogic_errors                                                                             │    _lakelogic_categories    │ quarantine_state │ quarantine_reprocessed │
│    int64    │    varchar    │         varchar         │    date     │ varchar │   double    │  boolean  │                                                                                varchar[]                                                                                 │          varcha

In [10]:
pl.read_parquet("data/silver/silver_customers.parquet")

customer_id,name,email,signup_date,tier,total_spend,is_active
i64,str,str,datetime[μs],str,f64,bool
101,"""Alice Smith""","""alice.smith@example.com""",2024-01-15 00:00:00,"""premium""",1250.5,true
104,"""Duplicate User""","""dup@example.com""",2024-01-18 00:00:00,"""free""",25.0,true


### Run Logs
Inspect recent LakeLogic runs from the local run log database.


In [11]:
from pathlib import Path
import duckdb

# Defaults come from contract metadata; adjust if you changed them.
log_db = Path("logs/lakelogic_run_logs.duckdb")
log_table = "lakelogic_run_logs"

if log_db.exists():
    con = duckdb.connect(str(log_db), read_only=True)
    try:
        query = f"""
        SELECT
           run_id, engine, domain, system, timestamp, contract, stage,
            counts_source, counts_total, counts_good, counts_quarantined,
            max_source_mtime, source_files_json
        FROM {log_table}
        ORDER BY timestamp DESC
        LIMIT 10
        """
        res = con.execute(query)
        try:
            import pandas as pd
            display(res.df())
        except Exception:
            for row in res.fetchall():
                print(row)
    finally:
        con.close()
else:
    print(f"Run log DB not found: {log_db.resolve()}")


,run_id,engine,domain,system,timestamp,contract,stage,counts_source,counts_total,counts_good,counts_quarantined,max_source_mtime,source_files_json
0,90335c18d86c40338b9178e12d1de922,duckdb,customer_analytics,crm_export,2026-02-08T10:08:54+00:00,Validated CRM customers,bronze,8,8,8,0,1.770460e+09,"[{""path"": ""D:\\Github\\_SaaS\\lakelogic\\examp..."
1,064195a67cfc4f4b9d0e1d57e1e5f8ef,duckdb,customer_analytics,crm_export,2026-02-08T10:08:54+00:00,Validated CRM customers,default,8,7,2,5,NaN,[]


## 7. Next Steps
Congratulations! You've just built a production-quality data gate.

**Try this:**
1. Edit `contract.yaml` to change the `bronze` stage path to a missing file and see how it fails gracefully.
2. Add a new CSV to the `data/` folder and watch the `bronze` stage pick it up automatically via the glob pattern!
